# Lopatica, rotor i vodomlazni pogon: predvidi → izračunaj → provjeri

Prvi pokus koristi **cijeli rotor** s punim stalnim sapničkim protokom Q.
Kasniji Z3 koristi **jednu pravocrtno gibajuću lopaticu** i zadani relativni
maseni dotok. Ti protoci nisu zamjenjivi.

## Predvidi prije računa

1. Zašto snaga cijeloga rotora nestaje pri u = 0 i pri u = c1?
2. Mijenja li stalni koeficijent k položaj optimuma ili njegovu vrijednost?
3. Može li proizvoljan izmjereni vektor sile odgovarati pasivnoj lopatici?
4. Doprinosi li radijalna komponenta brzine osnom momentu rotora iz Z4?
5. Koji kandidat Z5 pri jednakom potisku traži manje električne snage:
   veći protok s manjom razlikom brzina ili manji protok s većom razlikom?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def pelton(c1, u, Q, rho=998.0, k=0.90, outlet_deviation_deg=15.0):
    # k je omjer iznosa izlazne i ulazne relativne brzine.
    c1, u = np.broadcast_arrays(c1, u)
    if np.any(c1 <= 0) or np.any((u < 0) | (u > c1)) or not (0 <= k <= 1):
        raise ValueError("Treba vrijediti c1>0, 0≤u≤c1 i 0≤k≤1.")
    phi = np.deg2rad(outlet_deviation_deg)
    w1 = c1-u
    c2_t = u-k*w1*np.cos(phi)
    force_t = rho*Q*(c1-c2_t)
    power = force_t*u
    jet_power = 0.5*rho*Q*c1**2
    return force_t, power, power/jet_power

c1, Q, k, phi = 36.0, 0.045, 0.90, 15.0
u_grid = np.linspace(0, c1, 1001)
force, power, efficiency = pelton(c1, u_grid, Q, k=k, outlet_deviation_deg=phi)
i_opt = int(np.argmax(power))
u_opt_num = u_grid[i_opt]
u_opt_analytic = c1/2
print(f"Numerički optimum u = {u_opt_num:.3f} m/s = {u_opt_num/c1:.3f} c1")
print(f"P_max = {power[i_opt]/1000:.3f} kW; eta_mlaz→rotor = {efficiency[i_opt]:.3f}")


## Izračunaj: optimiranje i osjetljivost

Snagu tražimo pretragom mreže, bez unošenja poznatog optimuma u algoritam. Nakon toga mijenjamo koeficijent očuvanja relativne brzine \(k\) i odstupanje izlaza od idealnog okreta. To razdvaja **položaj optimuma** od **vrijednosti optimuma**.


In [ ]:
k_values = [0.80, 0.90, 1.00]
phi_values = np.linspace(0, 30, 61)
eta_max = np.empty((len(k_values), len(phi_values)))
for i, k_i in enumerate(k_values):
    for j, phi_i in enumerate(phi_values):
        eta_max[i,j] = pelton(c1, c1/2, Q, k=k_i, outlet_deviation_deg=phi_i)[2]

# Neovisna lokalna provjera stacionarnosti: centrirana derivacija snage.
du = 1e-3*c1
p_plus = pelton(c1, c1/2+du, Q, k=k, outlet_deviation_deg=phi)[1]
p_minus = pelton(c1, c1/2-du, Q, k=k, outlet_deviation_deg=phi)[1]
dP_du = (p_plus-p_minus)/(2*du)
print(f"Numerička derivacija dP/du u c1/2: {float(dP_du):.3e} N")
print(f"Pad eta_max zbog k: 1.00→0.80 pri φ=15°: {eta_max[-1,30]-eta_max[0,30]:.3f}")


## Provjeri

Provjere koriste optimalnost, granične slučajeve i energetsku granicu. U ovom pojednostavljenom modelu nema mehaničkih gubitaka rotora; zato dobivena učinkovitost nije ukupna učinkovitost turbine.


In [ ]:
assert np.isclose(u_opt_num, u_opt_analytic, atol=c1/1000)
assert abs(float(dP_du)) < 1e-7*power[i_opt]/c1
assert np.isclose(pelton(c1, 0.0, Q, k=k, outlet_deviation_deg=phi)[1], 0.0)
assert np.isclose(pelton(c1, c1, Q, k=k, outlet_deviation_deg=phi)[1], 0.0)
assert np.all((eta_max >= 0) & (eta_max <= 1+1e-12))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for k_i in k_values:
    p_i = pelton(c1, u_grid, Q, k=k_i, outlet_deviation_deg=phi)[1]
    axes[0].plot(u_grid/c1, p_i/1000, label=f"k={k_i:.2f}")
axes[0].axvline(.5, color="#b43c35", ls="--", label="u/c1=0,5")
axes[0].set(xlabel="$u/c_1$", ylabel="snaga (kW)", title="Numeričko traženje optimuma")
axes[0].legend()
for i, k_i in enumerate(k_values):
    axes[1].plot(phi_values, eta_max[i], label=f"k={k_i:.2f}")
axes[1].set(xlabel="odstupanje izlaza φ (°)", ylabel="najveća učinkovitost mlaza", title="Osjetljivost na gubitak i kut")
axes[1].legend()
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Z3: rekonstruiraj tok iz sile

Podatci su sintetički i jednaki objavljenom zadatku. Sila je sila **fluida
na lopaticu**, a 18 kg/s relativni maseni protok kroz gibajući kontrolni
volumen. Rješavamo linearni sustav za apsolutni izlaz. Energiju zatim
provjeravamo zasebno u apsolutnim i relativnim brzinama.

In [ ]:
c_in = np.array([32.0, 0.0])
u_blade = np.array([12.0, 0.0])
m_rel = 18.0
force_measured = np.array([625.0, -153.0])
c_out = np.linalg.solve(m_rel*np.eye(2), m_rel*c_in-force_measured)
w_in, w_out = c_in-u_blade, c_out-u_blade
k_reconstructed = np.linalg.norm(w_out)/np.linalg.norm(w_in)
beta = np.degrees(np.arctan2(w_out[1], w_out[0]))
P_blade = np.dot(force_measured, u_blade)
loss_absolute = m_rel*(np.dot(c_in,c_in)-np.dot(c_out,c_out))/2-P_blade
loss_relative = m_rel*(np.dot(w_in,w_in)-np.dot(w_out,w_out))/2
assert np.allclose(m_rel*(c_in-c_out),force_measured,atol=1e-10)
assert np.isclose(loss_absolute,loss_relative,rtol=0,atol=1e-9)
assert 0 < k_reconstructed < 1 and loss_absolute > 0
assert np.allclose(c_out,[-2.722222222222222,8.5],rtol=0,atol=1e-10)
assert np.isclose(P_blade,7500.0,rtol=0,atol=1e-9)
print(f'c2 = {c_out} m/s; w2 = {w_out} m/s')
print(f'k = {k_reconstructed:.6f}; beta = {beta:.5f} deg')
print(f'P = {P_blade/1000:.3f} kW; gubitak = {loss_absolute/1000:.6f} kW')

# Osjetljivost modela na alternativne sile; nije mjerna nesigurnost zadatka.
Fx_sweep = np.linspace(550,800,101)
c_sweep = np.column_stack([32-Fx_sweep/m_rel,np.full_like(Fx_sweep,8.5)])
w_sweep = c_sweep-u_blade
k_sweep = np.linalg.norm(w_sweep,axis=1)/np.linalg.norm(w_in)
loss_sweep = .5*m_rel*(np.dot(w_in,w_in)-np.sum(w_sweep*w_sweep,axis=1))
assert k_sweep[-1] > 1 and loss_sweep[-1] < 0
fig, ax = plt.subplots(figsize=(7,3.4))
ax.plot(Fx_sweep,k_sweep,label='rekonstruirani k')
ax.axhline(1,color='#c0392b',ls='--',label='granica pasivne lopatice')
ax.scatter([625],[k_reconstructed],color='#1565c0',label='zadani podatak')
ax.set(xlabel='Fx fluida na lopaticu (N)',ylabel='k',title='Koje sile dopušta pasivni model?')
ax.grid(ls=':',alpha=.4);ax.legend();plt.tight_layout();plt.show()

## Z4: dva lokalna trokuta brzina i Eulerov rad

Oba se lokalna prikaza crtaju s pozitivnim tangencijalnim smjerom udesno
i radijalnim prema gore. Vektorski zbroj ide od ishodišta preko kraja u
do kraja c: **c = u + w**. Kontrolni volumen obuhvaća cijeli rotor.
Pozitivan moment u računu jest moment rotora na fluid.

In [ ]:
radii = np.array([.060,.140])
omega, m_rotor = 200.0, 3.00
c_sections = np.array([[5.0,4.0],[20.0,6.0]])
u_sections = np.column_stack([omega*radii,np.zeros(2)])
w_sections = c_sections-u_sections
angular_momentum_flux = m_rotor*radii*c_sections[:,0]
M_rotor_fluid = np.diff(angular_momentum_flux).item()
work = np.diff(np.sum(u_sections*c_sections,axis=1)).item()
P_euler = m_rotor*work
assert np.allclose(u_sections+w_sections,c_sections,rtol=0,atol=1e-12)
assert np.allclose(w_sections,[[-7,4],[-8,6]],rtol=0,atol=1e-12)
assert np.isclose(M_rotor_fluid,7.5,rtol=0,atol=1e-12)
assert np.isclose(P_euler,omega*M_rotor_fluid,rtol=0,atol=1e-10)
assert np.isclose(work,500.0,rtol=0,atol=1e-10)
print(f'M rotor→fluid = {M_rotor_fluid:+.3f} N m; reakcija = {-M_rotor_fluid:+.3f} N m')
print(f'P rotor→fluid = {P_euler/1000:+.3f} kW; e = {work:+.3f} J/kg')
fig,axes=plt.subplots(1,2,figsize=(10,3.5))
for i,ax in enumerate(axes):
    for start,vec,label,color in [(np.zeros(2),u_sections[i],'u','#b7600c'),
                                 (u_sections[i],w_sections[i],'w','#1565c0'),
                                 (np.zeros(2),c_sections[i],'c','#1e8449')]:
        ax.annotate('',xy=start+vec,xytext=start,
                    arrowprops={'arrowstyle':'->','color':color,'lw':2})
        middle=start+vec/2
        ax.text(middle[0],middle[1]+.6,label,color=color)
    ax.set(xlim=(-2,31),ylim=(-2,10),xlabel='t (m/s)',ylabel='r (m/s)',
           title=f'Presjek {i+1}; r = {radii[i]:.3f} m',aspect='equal')
    ax.grid(ls=':',alpha=.4)
plt.tight_layout();plt.show()

## Z5: jednak potisak, različit protok i potrošnja

Ulazne i izlazne brzine prvo čitamo u brodskom okviru. Atmosferski tlakovi,
jednake geodetske visine i zanemareni dovodni gubitci daju račun iz zadatka.
Kinetička energija daleke trake provjerava se u okviru mirujuće vode.
Zadana učinkovitost 0,80 povezuje električnu snagu i snagu predanu fluidu;
propulzijska učinkovitost zasebno uspoređuje TU s tom snagom fluida.

In [ ]:
T, U, rho_water, eta_drive, P_limit = 2000.0, 8.0, 1000.0, .80, 40000.0
jet_speeds = np.array([20.0,30.0])
Q_candidates = T/(rho_water*(jet_speeds-U))
d_candidates = np.sqrt(4*Q_candidates/(np.pi*jet_speeds))
Ph_candidates = .5*rho_water*Q_candidates*(jet_speeds**2-U**2)
Pel_candidates = Ph_candidates/eta_drive
P_useful = T*U
P_wake = .5*rho_water*Q_candidates*(jet_speeds-U)**2
eta_prop = P_useful/Ph_candidates
feasible = Pel_candidates <= P_limit
assert np.allclose(rho_water*Q_candidates*(jet_speeds-U),T,rtol=0,atol=1e-9)
assert np.allclose(Ph_candidates,P_useful+P_wake,rtol=0,atol=1e-9)
assert np.allclose(Pel_candidates,[35000,47500],rtol=0,atol=1e-9)
assert feasible.tolist() == [True,False]
assert np.all((eta_prop>0)&(eta_prop<1))
for i,name in enumerate(['A','B']):
    print(f'{name}: Q={1000*Q_candidates[i]:.5f} L/s; d={1000*d_candidates[i]:.5f} mm; '
          f'Ph={Ph_candidates[i]/1000:.3f} kW; Pel={Pel_candidates[i]/1000:.3f} kW; '
          f'eta_prop={eta_prop[i]:.5f}; izvedivo={feasible[i]}')
speed_sweep=np.linspace(12,40,141)
flow_sweep=T/(rho_water*(speed_sweep-U))
electric_sweep=.5*rho_water*flow_sweep*(speed_sweep**2-U**2)/eta_drive
assert np.all(np.diff(electric_sweep)>0) and np.all(np.diff(flow_sweep)<0)
fig,ax=plt.subplots(figsize=(7,3.5))
ax.plot(speed_sweep,electric_sweep/1000,label='isti T = 2 kN')
ax.scatter(jet_speeds,Pel_candidates/1000,label='A i B')
ax.axhline(P_limit/1000,color='#c0392b',ls='--',label='dostupno 40 kW')
ax.set(xlabel='Vj prema brodu (m/s)',ylabel='električna snaga (kW)',title='Izbor uz ograničenje snage')
ax.legend();ax.grid(ls=':',alpha=.4);plt.tight_layout();plt.show()

## Protumači

1. Zašto puni sapnički protok daje optimum u/c1 = 1/2, a dotok jedne
   pravocrtno gibajuće lopatice drugi optimum? Koji model koristi prvi pokus?
2. Kako ograničenje u < c1/2 mijenja izvedivi optimum cijeloga rotora?
3. Što bi u Z3 značili k > 1 i negativan izračunati gubitak? Koje podatke
   ili pretpostavke treba provjeriti prije tvrdnje o pasivnoj lopatici?
4. Zašto u Z4 ne smijemo upotrijebiti isti u na oba radijusa? Koji moment
   prima rotor, a koji fluid? Može li radijalna brzina promijeniti taj moment?
5. Zašto se u Z5 mora oduzeti ulazni tok količine gibanja? U kojem graničnom slučaju nestaje?
6. Kandidat A traži veći protok, a manju snagu. Koji praktični uvjeti nisu
   obuhvaćeni ovim idealiziranim izborom (dovod, kavitacija, trag i gubitci)?
7. Pri U = 0 može postojati potisak uz TU = 0. Zašto to ne znači nultu
   potrebnu snagu pogona? Razlikuj dvije ovdje uporabljene učinkovitosti.